# EmoWave - SVM Pipeline (Local)

**Tai hien bai bao Wang et al. (2014):**
Power Spectrum + Asymmetry Features + LDS Smoothing + SVM

---

**Yeu cau:** File `s01.dat` -> `s32.dat` dat trong thu muc `data/deap/`

## 1. Cau hinh

In [ ]:
import os

# --- CAU HINH ---
PROCESSED_DIR = "../data/processed"       # Duong dan toi thu muc chua file .npy
USE_LDS = True                   # LDS smoothing
USE_GRIDSEARCH = True            # GridSearch (tat de chay nhanh)
SEARCH_RBF = False               # False: Chi GridSearch Linear (nhanh, khuyen dung). True: GridSearch ca RBF (RAT CHAM)
MODE = "subject-dependent"      # "subject-dependent": Train model rieng cho tung subject (khuyen dung, giong bai bao gốc)
                                 # "subject-independent": Train tren 31 subjects, test tren 1 subject (LOSO)
                                 # "subject-mixed": Tron du lieu tat ca subjects vao train chung
RESULTS_DIR = "../results"

# Kiem tra data
files = ["X_epochs.npy", "y_valence.npy", "y_arousal.npy", "subject_groups.npy"]
print(f"Kiem tra cac file trong {PROCESSED_DIR}:")
for f in files:
    path = os.path.join(PROCESSED_DIR, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  [OK] {f} ({size_mb:.1f} MB)")
    else:
        print(f"  [MISSING] {f}")

## 2. Import Libraries

In [ ]:
import numpy as np
import pickle
import json
import time
import gc
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)

# Khoi tao bien toan cuc de luu trong so model
global_weights = {}

print("All imports OK")

## 3. DEAP Loader

In [ ]:
SFREQ = 128

def load_processed_data():
    print(f"Loading data from {PROCESSED_DIR}...")
    X_epochs = np.load(os.path.join(PROCESSED_DIR, "X_epochs.npy"))
    y_valence = np.load(os.path.join(PROCESSED_DIR, "y_valence.npy"))
    y_arousal = np.load(os.path.join(PROCESSED_DIR, "y_arousal.npy"))
    subject_groups = np.load(os.path.join(PROCESSED_DIR, "subject_groups.npy"))
    
    print(f"Loaded X_epochs: {X_epochs.shape}")
    return X_epochs, y_valence, y_arousal, subject_groups

print("DEAP Loader ready")

## 4. Feature Extraction

In [ ]:
FREQ_BANDS = {
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45),
}

ASYMMETRY_PAIRS = [
    (0, 16), (1, 17), (2, 18), (3, 19), (4, 20), (5, 21), (6, 22),
    (7, 23), (8, 24), (9, 25), (10, 26), (11, 27), (12, 28), (13, 29),
]

def extract_features_single(epoch, sfreq=128):
    """Trich xuat features tu 1 epoch (32, 128).
    
    Tra ve vector gom:
      - PSD:        32 ch x N bands = 128 features
      - Asymmetry:  14 pairs x N bands = 56 features
      - DE:         32 ch x N bands = 128 features (Differential Entropy)
    Total: 312 features
    """
    from scipy.signal import welch
    from scipy.stats import skew, kurtosis
    
    n_channels = epoch.shape[0]
    bands = list(FREQ_BANDS.values())
    n_bands = len(bands)
    
    # --- PSD (Welch) ---
    freqs, psd = welch(epoch, fs=sfreq, nperseg=min(128, epoch.shape[1]))
    psd_features = []
    de_features = []
    for ch in range(n_channels):
        for (fmin, fmax) in bands:
            idx = (freqs >= fmin) & (freqs <= fmax)
            band_power = np.mean(psd[ch, idx]) if np.any(idx) else 1e-10
            psd_features.append(band_power)
            
            # --- Differential Entropy ---
            # DE = 0.5 * log(2 * pi * e * variance_of_band)
            band_var = np.var(psd[ch, idx]) if np.any(idx) else 1e-10
            de = 0.5 * np.log(2 * np.pi * np.e * max(band_var, 1e-10))
            de_features.append(de)
    
    # --- Asymmetry (Left - Right) ---
    asym_features = []
    for (left_ch, right_ch) in ASYMMETRY_PAIRS:
        for b_i, (fmin, fmax) in enumerate(bands):
            idx = (freqs >= fmin) & (freqs <= fmax)
            left_power = np.mean(psd[left_ch, idx]) if np.any(idx) else 1e-10
            right_power = np.mean(psd[right_ch, idx]) if np.any(idx) else 1e-10
            asym_features.append(np.log(max(left_power, 1e-10)) - np.log(max(right_power, 1e-10)))
    
    # --- Statistical Features (per channel) ---
    stat_features = []
    for ch in range(n_channels):
        stat_features.append(np.mean(epoch[ch]))
        stat_features.append(np.var(epoch[ch]))
        stat_features.append(float(skew(epoch[ch])))
        stat_features.append(float(kurtosis(epoch[ch])))
    
    return np.concatenate([psd_features, asym_features, de_features])

def extract_features_dataset(X_epochs):
    """Trich xuat features cho toan bo dataset."""
    n_epochs = X_epochs.shape[0]
    sample = extract_features_single(X_epochs[0])
    n_features = len(sample)
    print(f"    Trich xuat {n_features} features/epoch x {n_epochs} epochs...")
    
    features = np.zeros((n_epochs, n_features), dtype=np.float32)
    features[0] = sample
    
    for i in range(1, n_epochs):
        features[i] = extract_features_single(X_epochs[i])
        if (i + 1) % 10000 == 0:
            print(f"    ...{i+1}/{n_epochs} epochs")
    
    print(f"    Done! Feature matrix shape: {features.shape}")
    return features

print("Feature extractors ready (PSD + Asymmetry + DE = 312 features)")


## 5. LDS Smoothing

In [ ]:
def lds_smoothing(features_seq, alpha=0.3):
    smoothed = np.zeros_like(features_seq)
    smoothed[0] = features_seq[0]
    for t in range(1, len(features_seq)):
        smoothed[t] = alpha * features_seq[t] + (1 - alpha) * smoothed[t - 1]
    return smoothed


def apply_lds(X_features, epochs_per_trial=60, alpha=0.3):
    X_smoothed = np.copy(X_features)
    n_trials = len(X_features) // epochs_per_trial
    for t in range(n_trials):
        s, e = t * epochs_per_trial, (t + 1) * epochs_per_trial
        X_smoothed[s:e] = lds_smoothing(X_features[s:e], alpha)
    return X_smoothed

print("LDS ready")

## 6. SVM Training

In [ ]:
from sklearn.decomposition import PCA

USE_PCA = False  # Bat PCA de giam chieu va loc nhieu

def train_svm_simple(X_train, y_train, kernel="linear", C=1.0):
    """Train SVM don gian. Dung LinearSVC cho linear kernel (nhanh gap 50x)."""
    steps = [('scaler', StandardScaler())]
    if USE_PCA:
        steps.append(('pca', PCA(n_components=0.95, random_state=42)))
    
    if kernel == "linear":
        steps.append(('svm', LinearSVC(C=C, class_weight='balanced', max_iter=5000, random_state=42, dual='auto')))
    else:
        steps.append(('svm', SVC(kernel=kernel, C=C, class_weight='balanced', random_state=42)))
    
    pipeline = Pipeline(steps)
    pipeline.fit(X_train, y_train)
    return pipeline

def train_svm_gridsearch(X_train, y_train):
    """GridSearchCV voi Linear kernel. Them PCA(95%) truoc SVM."""
    steps = [('scaler', StandardScaler())]
    if USE_PCA:
        steps.append(('pca', PCA(n_components=0.95, random_state=42)))
    steps.append(('svm', LinearSVC(class_weight='balanced', max_iter=5000, random_state=42, dual='auto')))
    
    linear_pipe = Pipeline(steps)
    linear_grid = GridSearchCV(
        linear_pipe, {'svm__C': [0.01, 0.1, 1, 10, 100]},
        cv=StratifiedKFold(5, shuffle=True, random_state=42),
        scoring='f1_macro', n_jobs=-1, verbose=0, refit=True
    )
    t0 = time.time()
    linear_grid.fit(X_train, y_train)
    
    pca_info = ""
    if USE_PCA:
        best_pipe = linear_grid.best_estimator_
        n_components = best_pipe.named_steps['pca'].n_components_
        explained = best_pipe.named_steps['pca'].explained_variance_ratio_.sum()
        pca_info = f" | PCA: {n_components} components ({explained*100:.1f}% variance)"
    
    print(f"    Best C={linear_grid.best_params_['svm__C']}"
          f" -> {linear_grid.best_score_*100:.2f}% ({time.time()-t0:.1f}s){pca_info}")

    if not SEARCH_RBF:
        return linear_grid.best_estimator_, linear_grid.best_params_, linear_grid.best_score_

    # RBF search (optional)
    steps_rbf = [('scaler', StandardScaler())]
    if USE_PCA:
        steps_rbf.append(('pca', PCA(n_components=0.95, random_state=42)))
    steps_rbf.append(('svm', SVC(class_weight='balanced', random_state=42)))
    
    rbf_pipe = Pipeline(steps_rbf)
    rbf_grid = GridSearchCV(
        rbf_pipe,
        {'svm__C': [0.1, 1, 10], 'svm__gamma': ['scale', 'auto']},
        cv=StratifiedKFold(5, shuffle=True, random_state=42),
        scoring='f1_macro', n_jobs=-1, verbose=0, refit=True
    )
    t0 = time.time()
    rbf_grid.fit(X_train, y_train)
    print(f"    RBF Best: {rbf_grid.best_params_}"
          f" -> {rbf_grid.best_score_*100:.2f}% ({time.time()-t0:.1f}s)")

    if rbf_grid.best_score_ > linear_grid.best_score_:
        print("    >> RBF thang Linear!")
        return rbf_grid.best_estimator_, rbf_grid.best_params_, rbf_grid.best_score_
    else:
        print("    >> Linear thang RBF!")
        return linear_grid.best_estimator_, linear_grid.best_params_, linear_grid.best_score_

print("SVM trainers ready (with PCA 95%)")


## 7. Evaluation

In [ ]:
def evaluate_and_plot(y_test, y_pred, label_type="2class",
                      best_params=None, cv_score=None):
    os.makedirs(RESULTS_DIR, exist_ok=True)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, labels=[0,1] if label_type=='2class' else [0,1,2,3], average='macro', zero_division=0)

    names = ["Negative", "Positive"] if label_type == "2class" \
            else ["Vui ve", "So hai", "Buon", "Thu gian"]

    print(f"\n  {'='*40}")
    print(f"  SVM --- {label_type} ({MODE} mode)")
    print(f"  {'='*40}")
    print(f"  Accuracy:      {acc*100:.2f}%")
    print(f"  F1 (weighted): {f1*100:.2f}%")
    if best_params:
        if isinstance(best_params, list):
            print(f"  Best params:   (Average / Multiple models across subjects)")
        else:
            print(f"  Best params:   {best_params}")
    print(classification_report(y_test, y_pred, labels=[0,1] if label_type=="2class" else [0,1,2,3], target_names=names, zero_division=0))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred, labels=[0,1] if label_type=="2class" else [0,1,2,3])
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title(f'SVM Confusion Matrix ({label_type} - {MODE})\n'
                 f'Accuracy: {acc*100:.2f}%', fontsize=14)
    plt.tight_layout()

    fig_path = os.path.join(RESULTS_DIR, f"confusion_matrix_svm_{label_type}.png")
    plt.savefig(fig_path, dpi=150)
    plt.show()

    # JSON
    results = {
        "model": "SVM", "label_type": label_type, "mode": MODE,
        "accuracy": round(acc, 4), "f1_macro": round(f1, 4),
        "best_params": str(best_params) if best_params else None,
        "cv_score": round(cv_score, 4) if cv_score else None,
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(
            y_test, y_pred, labels=[0,1] if label_type=="2class" else [0,1,2,3], target_names=names, output_dict=True, zero_division=0),
    }
    json_path = os.path.join(RESULTS_DIR, f"svm_results_{label_type}.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"  Saved: {fig_path}")
    print(f"  Saved: {json_path}")
    return results

print("Evaluation ready")

## 8. Unified Pipeline Execution Function

In [ ]:
def get_labels_from_val_aro(y_val, y_aro, label_type):
    if label_type == "2class":
        return y_val
    else:
        # 4-class logic mapping:
        # val=1, aro=1 -> 0 (Vui ve)
        # val=0, aro=1 -> 1 (So hai)
        # val=0, aro=0 -> 2 (Buon)
        # val=1, aro=0 -> 3 (Thu gian)
        classes = np.zeros(len(y_val), dtype=int)
        classes[(y_val == 1) & (y_aro == 1)] = 0
        classes[(y_val == 0) & (y_aro == 1)] = 1
        classes[(y_val == 0) & (y_aro == 0)] = 2
        classes[(y_val == 1) & (y_aro == 0)] = 3
        return classes

def run_pipeline(label_type="2class"):
    print("=" * 60)
    print(f"  EmoWave SVM Pipeline --- {label_type} ({MODE} mode)")
    print("=" * 60)
    
    # 1. Load Data
    print("\n[1] Loading Preprocessed Data...")
    X_epochs, y_val, y_aro, groups = load_processed_data()
    y_all = get_labels_from_val_aro(y_val, y_aro, label_type)
    
    if MODE == "subject-independent":
        # ============================================================
        #  LOSO CHUAN: Loop qua tat ca 32 subjects
        #  Moi vong: Train tren 31 nguoi, Test tren 1 nguoi
        # ============================================================
        unique_groups = np.unique(groups)
        n_subjects = len(unique_groups)
        print(f"\n[2] LOSO Mode: {n_subjects} folds (moi fold bo ra 1 subject)")
        
        # 2a. Trich xuat features 1 lan duy nhat (an toan vi PSD la per-epoch)
        print("\n[3] Feature extraction (toan bo data)...")
        X_feat_all = extract_features_dataset(X_epochs)
        del X_epochs; gc.collect()
        
        if USE_LDS:
            print("\n[4] LDS smoothing...")
            X_feat_all = apply_lds(X_feat_all)
        
        # 2b. Chay LOSO 32 vong
        print(f"\n[5] Bat dau LOSO {n_subjects} folds...")
        t_start = time.time()
        
        y_test_all, y_pred_all = [], []
        skipped_folds = []
        fold_results = []
        subject_weights = []
        
        for fold_i, test_group in enumerate(unique_groups):
            train_idx = (groups != test_group)
            test_idx = (groups == test_group)
            
            X_train = X_feat_all[train_idx]
            y_train = y_all[train_idx]
            X_test = X_feat_all[test_idx]
            y_test = y_all[test_idx]
            
            # Edge case: Subject chi co 1 class -> skip
            if len(np.unique(y_test)) < 2:
                skipped_folds.append(int(test_group))
                print(f"  Fold {fold_i+1:02d}/32 (S{int(test_group)+1:02d}): SKIP - chi co 1 class trong test set")
                continue
            
            if len(np.unique(y_train)) < 2:
                skipped_folds.append(int(test_group))
                print(f"  Fold {fold_i+1:02d}/32 (S{int(test_group)+1:02d}): SKIP - chi co 1 class trong train set")
                continue
            
            # Train SVM (scaler duoc khoi tao moi trong pipeline moi fold)
            if USE_GRIDSEARCH:
                model, params, cv_score = train_svm_gridsearch(X_train, y_train)
            else:
                model = train_svm_simple(X_train, y_train)
                params, cv_score = None, None
            
            try:
                if not USE_PCA:
                    svm_step = model.named_steps['svm']
                    if hasattr(svm_step, 'coef_'):
                        subject_weights.append(svm_step.coef_[0])
            except:
                pass
            
            # Predict
            y_pred = model.predict(X_test)
            
            fold_acc = accuracy_score(y_test, y_pred)
            fold_f1 = f1_score(y_test, y_pred, 
                              labels=[0,1] if label_type=='2class' else [0,1,2,3],
                              average='macro', zero_division=0)
            
            fold_results.append({
                'subject': int(test_group),
                'acc': fold_acc,
                'f1': fold_f1,
                'n_test': len(y_test)
            })
            
            y_test_all.append(y_test)
            y_pred_all.append(y_pred)
            
            elapsed = time.time() - t_start
            eta = elapsed / (fold_i + 1) * (n_subjects - fold_i - 1)
            print(f"  Fold {fold_i+1:02d}/32 (S{int(test_group)+1:02d}): "
                  f"Acc={fold_acc*100:.1f}% F1={fold_f1*100:.1f}% "
                  f"[{elapsed:.0f}s elapsed, ~{eta:.0f}s remaining]")
        
        del X_feat_all; gc.collect()
        
        total_time = time.time() - t_start
        print(f"\n  >> LOSO hoan tat! {len(fold_results)}/{n_subjects} folds thanh cong. "
              f"Tong thoi gian: {total_time:.1f}s ({total_time/60:.1f} phut)")
        
        if skipped_folds:
            print(f"  >> Folds bi skip (chi 1 class): {skipped_folds}")
        
        # In ket qua tung fold
        print(f"\n  {'Subject':>8} {'Acc':>8} {'F1':>8}")
        print(f"  {'---':>8} {'---':>8} {'---':>8}")
        for fr in fold_results:
            print(f"  S{fr['subject']+1:02d}{fr['acc']*100:>8.2f}%{fr['f1']*100:>8.2f}%")
        
        # Tinh trung binh
        if fold_results:
            mean_acc = np.mean([fr['acc'] for fr in fold_results])
            mean_f1 = np.mean([fr['f1'] for fr in fold_results])
            std_acc = np.std([fr['acc'] for fr in fold_results])
            std_f1 = np.std([fr['f1'] for fr in fold_results])
            print(f"  {'---':>8} {'---':>8} {'---':>8}")
            print(f"  {'Mean':>8}{mean_acc*100:>8.2f}%{mean_f1*100:>8.2f}%")
            print(f"  {'Std':>8}{std_acc*100:>8.2f}%{std_f1*100:>8.2f}%")
        
        # Ghep tat ca predictions de ve Confusion Matrix tong
        if y_test_all:
            y_test_combined = np.concatenate(y_test_all)
            y_pred_combined = np.concatenate(y_pred_all)
            
            # Luu ket qua LOSO chi tiet vao JSON
            results = evaluate_and_plot(y_test_combined, y_pred_combined, label_type)
            results['loso_fold_results'] = fold_results
            results['loso_skipped_folds'] = skipped_folds
            results['loso_mean_acc'] = round(mean_acc, 4)
            results['loso_mean_f1'] = round(mean_f1, 4)
            results['loso_std_acc'] = round(std_acc, 4)
            results['loso_std_f1'] = round(std_f1, 4)
            
            # Ghi de JSON voi thong tin LOSO
            json_path = os.path.join(RESULTS_DIR, f"svm_results_{label_type}.json")
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            
            return results
        else:
            print("  ERROR: Khong co fold nao thanh cong!")
            return None
        
    else:
        # subject-dependent
        unique_groups = np.unique(groups)
        print(f"\n[2] Huan luyen models rieng biet cho {len(unique_groups)} subjects...")
        t_start = time.time()
        
        y_test_all, y_pred_all, best_params_all, subject_weights = [], [], [], []
        
        for sg in unique_groups:
            idx = (groups == sg)
            X_sub_ep = X_epochs[idx]
            y_sub = y_all[idx]
            
            if len(np.unique(y_sub)) < 2:
                print(f"  Subject {sg:02d}: Bo qua vi chi co 1 nhan")
                continue
                
            X_sub_feat = extract_features_dataset(X_sub_ep)
            if USE_LDS:
                X_sub_feat = apply_lds(X_sub_feat)
                
            X_train, X_test, y_train, y_test = train_test_split(
                X_sub_feat, y_sub, test_size=0.2, random_state=42, stratify=y_sub)
            
            if USE_GRIDSEARCH:
                model, params, _ = train_svm_gridsearch(X_train, y_train)
                best_params_all.append(params)
            else:
                model = train_svm_simple(X_train, y_train)
                
            try:
                if not USE_PCA:
                    svm_step = model.named_steps['svm']
                    if hasattr(svm_step, 'coef_'):
                        subject_weights.append(svm_step.coef_[0])
            except:
                pass
            
            # Dung model.predict() de chay toan bo pipeline (Scaler -> PCA -> SVM)
            y_pred = model.predict(X_test)
            
            y_test_all.append(y_test)
            y_pred_all.append(y_pred)
            acc = accuracy_score(y_test, y_pred)
            print(f"  Subject {sg:02d}: Acc = {acc*100:.2f}%")
            
        y_test_combined = np.concatenate(y_test_all, axis=0)
        y_pred_combined = np.concatenate(y_pred_all, axis=0)
        print(f"\n  >> Da xong {len(unique_groups)} subjects! Tong thoi gian: {time.time()-t_start:.1f}s")
        
        if len(subject_weights) > 0:
            global_weights[label_type] = np.mean(subject_weights, axis=0)
            
        results = evaluate_and_plot(y_test_combined, y_pred_combined, label_type, best_params_all)
        return results

print("run_pipeline function ready")


---
## 9. CHAY PIPELINE - 2 CLASS (Positive / Negative)

In [ ]:
results_2c = run_pipeline(label_type="2class")

## 10. CHAY PIPELINE - 4 CLASS (Vui / So / Buon / Thu gian)

In [ ]:
results_4c = run_pipeline(label_type="4class")

## 11. Tong ket

In [ ]:
print("=" * 60)
print(f"  SVM Pipeline DONE! Mode: {MODE}")
print("=" * 60)
print(f"\n  {'Label':<10} {'Accuracy':>10} {'F1':>10}")
print(f"  {'---':<10} {'---':>10} {'---':>10}")
print(f"  {'2class':<10} {results_2c['accuracy']*100:>9.2f}% {results_2c['f1_macro']*100:>9.2f}%")
print(f"  {'4class':<10} {results_4c['accuracy']*100:>9.2f}% {results_4c['f1_macro']*100:>9.2f}%")
print(f"\n  Results saved in: {os.path.abspath(RESULTS_DIR)}/")

## 12. Feature Importance Analysis (Do quan trong cua vung nao)

In [ ]:
if "2class" in global_weights:
    bands = list(FREQ_BANDS.keys())
    n_bands = len(bands)
    channels = [
        'Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz',
        'Fp2', 'AF4', 'F4', 'F8', 'FC6', 'FC2', 'C4', 'T8', 'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2', 'FCz', 'Cz'
    ]
    asym_pairs = [
        ('Fp1', 'Fp2'), ('AF3', 'AF4'), ('F3', 'F4'), ('F7', 'F8'), 
        ('FC5', 'FC6'), ('FC1', 'FC2'), ('C3', 'C4'), ('T7', 'T8'), 
        ('CP5', 'CP6'), ('CP1', 'CP2'), ('P3', 'P4'), ('P7', 'P8'), 
        ('PO3', 'PO4'), ('O1', 'O2')
    ]
    stat_names = ['mean', 'var', 'skew', 'kurtosis']

    feature_names = []
    # PSD features
    for ch in channels:
        for b in bands:
            feature_names.append(f"PSD_{ch}_{b}")
    # Asymmetry features
    for left, right in asym_pairs:
        for b in bands:
            feature_names.append(f"ASYM_{left}-{right}_{b}")
    # DE features
    for ch in channels:
        for b in bands:
            feature_names.append(f"DE_{ch}_{b}")


    svm_coef = global_weights["2class"]
    
    if USE_PCA:
        print("Feature importance khong truc tiep kha dung khi bat PCA.")
        print(f"PCA da giam tu {len(feature_names)} features xuong {len(svm_coef)} components.")
        print("De xem feature importance, hay tat USE_PCA = False va chay lai.")
    elif len(svm_coef) != len(feature_names):
        print(f"WARNING: Trong so co {len(svm_coef)} features nhung feature_names co {len(feature_names)}.")
    else:
        sorted_idx = np.argsort(svm_coef)
        print("=" * 60)
        print("  DO QUAN TRONG CUA CAC DAC TRUNG EEG (2-CLASS)")
        print("=" * 60)
        print(f"\n  Tong so features: {len(feature_names)}")
        print(f"    PSD: {len(channels) * n_bands} | Asymmetry: {len(asym_pairs) * n_bands}")
        print(f"    DE: {len(channels) * n_bands}")
        
        print("\n[+] Top 10 dac trung ung ho cam xuc tich cuc (Positive):")
        for idx in sorted_idx[-10:][::-1]:
            print(f"  {feature_names[idx]:<30}: Trong so = {svm_coef[idx]:.4f}")

        print("\n[-] Top 10 dac trung ung ho cam xuc tieu cuc (Negative):")
        for idx in sorted_idx[:10]:
            print(f"  {feature_names[idx]:<30}: Trong so = {svm_coef[idx]:.4f}")
else:
    print("Vui long chay xong Pipeline 2-class (kernel Linear) truoc.")